<a href="https://colab.research.google.com/github/dnevo/clusters_unmixing/blob/main/notebooks/experiment_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clusters unmixing experiments

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dnevo/clusters_unmixing/blob/main/notebooks/experiment_review.ipynb)

This notebook runs the configured spectral unmixing experiments and presents the results in a notebook-friendly review format.

It covers:
- loading the project configuration and executing the configured experiment runs
- inspecting raw and normalized cluster spectra for each run
- comparing cosine off-diagonal statistics across preprocessing steps
- reviewing model metrics for the selected unmixing methods
- previewing predicted abundances against the synthetic ground truth
- visualizing synthetic pixel spectra generated for each run

In [ ]:
import subprocess
import sys
from pathlib import Path

WITH_MAMBA = False
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    project_root = Path("/content/clusters_unmixing")
    if not project_root.exists():
        subprocess.check_call([
            "git",
            "clone",
            "https://github.com/dnevo/clusters_unmixing.git",
            str(project_root),
        ])
else:
    NOTEBOOK_DIR = Path.cwd()
    project_root = NOTEBOOK_DIR.parent

SRC_DIR = project_root / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

GH_TOKEN = None
if IN_COLAB:
    from google.colab import userdata # type: ignore
    try:
        GH_TOKEN = userdata.get("GH_TOKEN")
    except Exception:
        GH_TOKEN = None  # not the owner's account, or access not granted

CAN_PUSH = bool(GH_TOKEN)
print("Colab:", IN_COLAB, "| will push:", CAN_PUSH)

In [ ]:
if IN_COLAB and WITH_MAMBA:
    # Install mamba and dependencies from wheels in Google Drive.Solution for Installation hangs in:
    # https://github.com/state-spaces/mamba/issues/983
    from google.colab import drive # type: ignore
    drive.mount('/content/drive')

    ! pip install /content/drive/MyDrive/cuda_wheels/causal_conv1d-1.6.2.post1-cp312-cp312-linux_x86_64.whl

    ! pip install /content/drive/MyDrive/cuda_wheels/mamba_ssm-2.3.2.post1-cp312-cp312-linux_x86_64.whl

In [ ]:
from clusters_unmixing.utils import run_experiments_notebook
run_experiments_notebook(project_root=project_root)

In [ ]:
import base64, requests

OWNER, REPO, BRANCH = "dnevo", "clusters_unmixing", "main"
LOCAL_DIR = project_root / "experiments" / "outputs"
DEST_DIR  = "experiments/outputs"   # path inside the repo

def push_file_to_github(local_file, dest_path, message="Update experiment outputs from Colab"):
    url = f"https://api.github.com/repos/{OWNER}/{REPO}/contents/{dest_path}"
    headers = {
        "Authorization": f"Bearer {GH_TOKEN}",
        "Accept": "application/vnd.github+json",
    }

    with open(local_file, "rb") as f:
        content = base64.b64encode(f.read()).decode()

    # The current blob SHA is required if the file already exists
    r = requests.get(url, headers=headers, params={"ref": BRANCH})
    sha = r.json().get("sha") if r.status_code == 200 else None

    payload = {"message": message, "content": content, "branch": BRANCH}
    if sha:
        payload["sha"] = sha

    r = requests.put(url, headers=headers, json=payload)
    r.raise_for_status()
    return r.json()["content"]["html_url"]

if CAN_PUSH:
    local_files = sorted(f for f in LOCAL_DIR.iterdir() if f.is_file())
    for local_file in local_files:
        dest_path = f"{DEST_DIR}/{local_file.name}"
        print("Pushed:", push_file_to_github(local_file, dest_path))
else:
    print("Skipping push (not in Colab, or no token).")

In [ ]:
if IN_COLAB:
    # disconnect from the runtime to avoid wasting resources when the notebook is not in use.
    from google.colab import runtime  # type: ignore[import-not-found]
    runtime.unassign()